# Session 13: Image Classification and Computer Vision Applications

**Learning goals:**

- Load labelled image files and prepare consistent CNN input tensors.
- Create training, validation and test batches with appropriate transforms and shuffle settings.
- Implement and evaluate a CNN classifier for cat and dog images.
- Distinguish image classification, object detection and image generation.
- Explain the basic ideas behind YOLO, DETR, GANs and diffusion models.

**Prerequisites:**

- Sessions 11–12: basic Python, CNNs and model training.
- Python with PyTorch, torchvision, Pillow, NumPy and Matplotlib.
- Run from the AI_SS course folder with the supplied `sample_data/` files; download the Section 4 dataset when needed.

This session connects image preparation to a complete classification workflow.
Work through the two short companion notebooks first, then apply those skills to colour photographs.
The final two sections provide a brief introduction to detection and generation.

## Outline

1. Loading Images with Datasets and Transforms
2. Preparing Image Batches for a CNN Model
3. Image Classification Workflow
4. Case Study: Dog and Cat Image Classification
5. Object Detection: YOLO and DETR
6. Image Generation: GANs and Diffusion Models

## 1. Loading Images with Datasets and Transforms

ImageFolder pairs each image with its class label. Grayscale, Resize and ToTensor
convert each image into a consistent `[1, 28, 28]` tensor, ready for batching and a CNN.

[Session 13.1: Loading Images with Datasets and Transforms](<Session13.1 Loading Images with Datasets and Transforms.ipynb>).

![Image files are resized and converted to single-channel tensors, then batched for a CNN.](assets/session13-image-to-cnn-pipeline.png)

## 2. Preparing Image Batches for a CNN Model

DataLoader groups image–label pairs into batches, shuffling their order for training.
Here, `batch_size=2` means two images per batch: four images form two batches.
A loop sends one batch at a time to the CNN until all samples have been processed.

[Session 13.2: Preparing Image Batches for a CNN Model](<Session13.2 Preparing Image Batches for a CNN Model.ipynb>).

![Four labelled images are shuffled into two clearly separated batches of two images. Batch 1 contains digits 1 and 0 with matching labels and is passed to the CNN before the next batch.](assets/session13-dataloader-batches.png)


## 3. Image Classification Workflow

Apply fixed image transforms, split the data into training and testing sets,
then batch each set. Train the CNN on training batches, evaluate the trained model
on testing batches using loss, accuracy and F1, then save the trained weights.

![Multiple labelled cat and dog images become tensors, which are split into training and testing sets. Each set is batched; training updates CNN weights, while testing measures loss, accuracy and F1 without weight updates. The trained model is then saved.](assets/session13-image-classification-workflow.png)


## 4. Case Study: Dog and Cat Image Classification

### 4.1 Obtain and organise the images

Dataset: [Dog vs Cat by Anthony Therrien on Kaggle](https://www.kaggle.com/datasets/anthonytherrien/dog-vs-cat).

1. Open the dataset page, review its description and usage terms, and download the archive.
2. Extract it and inspect the actual folder names. Do not assume the archive already uses the teaching layout below.
3. For a single labelled image pool, place cat photos in `sample_data/dog_vs_cat/cat/` and dog photos in `sample_data/dog_vs_cat/dog/`.
4. If the download supplies labelled training, validation and test sets, preserve their membership.
   Arrange them under `train/`, `val/` and `test/`, each with `cat/` and `dog/`, and set `DATA_LAYOUT = "split_folders"` below.
   Rename a `validation/` folder to `val/` in your working copy if needed.
5. If only some official splits exist, finish preparing three labelled, disjoint splits from the training pool before using `split_folders`.
   Unlabelled test images cannot be used to calculate test accuracy.

Single-pool teaching layout:

```text
sample_data/dog_vs_cat/
    cat/
        photo_001.jpg
        ...
    dog/
        photo_001.jpg
        ...
```

Preserved-split teaching layout:

```text
sample_data/dog_vs_cat/
    train/cat/    train/dog/
    val/cat/      val/dog/
    test/cat/     test/dog/
```

Folder names are class labels; do not infer them from arbitrary filenames. Inspect labels
before copying files. Kaggle images are not bundled with this notebook, and the notebook
does not download data or require API credentials. The first code cell reports a missing
folder clearly until you complete these steps.

### 4.2 Configure the experiment and transforms

Start with a balanced subset and three epochs to check the workflow. This is a baseline,
not a promised accuracy target. Keep the seed, split and settings when comparing models.
Exact numerical reproducibility can vary across devices and library versions.

[Official documentation: ImageFolder](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.ImageFolder.html)
· [Transforms](https://docs.pytorch.org/vision/stable/transforms.html)


In [ ]:
import os
import random
import numpy as np
import torch
from torchvision import datasets, transforms

DATA_ROOT = "sample_data/dog_vs_cat"
DATA_LAYOUT = "class_folders"  # Use "split_folders" for prepared train/val/test folders.
SEED = 42
IMAGE_SIZE = (64, 64)
BATCH_SIZE = 32
MAX_IMAGES_PER_CLASS = 500  # Cap each class in each input pool for a classroom run; use None for all.
EPOCHS = 3
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if not os.path.isdir(DATA_ROOT):
    raise FileNotFoundError("Prepare the Kaggle image folders described in Section 4.1: " + DATA_ROOT)

train_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),  # Standardise the height and width.
    transforms.RandomHorizontalFlip(p=0.5),  # Augment training images only.
    transforms.ToTensor(),  # Keep all three RGB channels and scale uint8 pixels to [0, 1].
])
evaluation_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),  # Use the same model input size.
    transforms.ToTensor(),  # Keep evaluation deterministic.
])
print("Device:", device, "Image size:", IMAGE_SIZE)


### 4.3 Inspect the image index and check readability

ImageFolder converts loaded images to RGB by default. Check all candidate image files
before training so corrupt images do not interrupt a later epoch. A readable image can
still have the wrong label, so visual inspection is also necessary.


In [ ]:
from PIL import Image

if DATA_LAYOUT == "class_folders":
    pool_roots = {"pool": DATA_ROOT}
elif DATA_LAYOUT == "split_folders":
    pool_roots = {name: os.path.join(DATA_ROOT, name) for name in ["train", "val", "test"]}
else:
    raise ValueError("DATA_LAYOUT must be class_folders or split_folders")

catalogues = {}
for name, folder in pool_roots.items():
    catalogue = datasets.ImageFolder(folder)  # Index paths and labels without an image transform.
    if catalogue.classes != ["cat", "dog"]:
        raise ValueError(f"Expected cat and dog class folders in {folder}; found {catalogue.classes}")
    unreadable = []
    for image_path, _ in catalogue.samples:
        try:
            with Image.open(image_path) as candidate:
                candidate.convert("RGB").load()  # Decode the complete image now.
        except (OSError, ValueError) as error:
            unreadable.append((image_path, str(error)))
    if unreadable:
        raise ValueError(f"Repair or remove unreadable images in your working copy: {unreadable[:3]}")
    catalogues[name] = catalogue
    counts = {label: catalogue.targets.count(index) for index, label in enumerate(catalogue.classes)}
    print(name, "class counts:", counts)
class_names = ["cat", "dog"]


### 4.4 Create training, validation and test subsets

For one image pool, take a seeded subset of each class and split it approximately 70/15/15.
Both classes appear in all three splits. Prepared splits retain their original membership.
Each split gets its own ImageFolder instance so training augmentation cannot affect validation.


In [ ]:
from torch.utils.data import DataLoader, Subset

split_rng = random.Random(SEED)

def choose_indices(catalogue):
    chosen = []
    for target in range(len(catalogue.classes)):
        indices = [i for i, label in enumerate(catalogue.targets) if label == target]
        split_rng.shuffle(indices)  # Reproduce subset selection from the experiment seed.
        if MAX_IMAGES_PER_CLASS is not None:
            indices = indices[:MAX_IMAGES_PER_CLASS]
        chosen.extend(indices)
    return chosen

if DATA_LAYOUT == "class_folders":
    pool = catalogues["pool"]
    chosen = choose_indices(pool)
    split_indices = {name: [] for name in ["train", "val", "test"]}
    for target in range(2):
        indices = [i for i in chosen if pool.targets[i] == target]
        if len(indices) < 10:
            raise ValueError("Provide at least 10 images per class for this example.")
        train_end = int(0.70 * len(indices))
        val_end = train_end + max(1, int(0.15 * len(indices)))
        split_indices["train"].extend(indices[:train_end])
        split_indices["val"].extend(indices[train_end:val_end])
        split_indices["test"].extend(indices[val_end:])
    split_roots = {name: DATA_ROOT for name in split_indices}
else:
    split_indices = {name: choose_indices(catalogue) for name, catalogue in catalogues.items()}
    split_roots = pool_roots

image_datasets, loaders = {}, {}
for name in ["train", "val", "test"]:
    transform = train_transform if name == "train" else evaluation_transform
    image_folder = datasets.ImageFolder(split_roots[name], transform=transform)
    image_datasets[name] = Subset(image_folder, split_indices[name])
    loaders[name] = DataLoader(
        image_datasets[name], batch_size=BATCH_SIZE, shuffle=(name == "train"),
        generator=torch.Generator().manual_seed(SEED), num_workers=0,
    )
    split_counts = {label: sum(image_folder.targets[i] == target for i in split_indices[name])
                    for target, label in enumerate(class_names)}
    if min(split_counts.values()) == 0:
        raise ValueError(f"Both classes must appear in {name}.")
    print(name, "images:", len(image_datasets[name]), "class counts:", split_counts)


### 4.5 Check for exact duplicates across splits

Hash decoded RGB pixels to catch identical images, including copies with different filenames.
Stop if the same pixels appear in different splits; remove duplicates or assign each duplicate
group to just one split before rerunning. This check does not detect near-duplicates, resized
copies or different photos of the same animal, so inspect those separately.


In [ ]:
import hashlib

image_owners = {}
for name, subset in image_datasets.items():
    for index in subset.indices:
        image_path, _ = subset.dataset.samples[index]
        with Image.open(image_path) as image:
            rgb = image.convert("RGB")
            payload = str(rgb.size).encode("ascii") + rgb.tobytes()
            fingerprint = hashlib.sha256(payload).hexdigest()  # Compare image contents, not filenames.
        previous = image_owners.get(fingerprint)
        if previous is not None and previous != name:
            raise ValueError(f"Duplicate image crosses {previous} and {name}: {image_path}")
        image_owners[fingerprint] = name
print("No exact decoded-image duplicates cross the selected splits.")


### 4.6 Inspect a training batch

Check that the images and labels agree. The shape should be `[batch, 3, 64, 64]`.
Matplotlib expects `[height, width, channels]`, so permute the channel dimension for display.


In [ ]:
import matplotlib.pyplot as plt

batch_images, batch_labels = next(iter(loaders["train"]))
print("Image shape:", tuple(batch_images.shape), "Label shape:", tuple(batch_labels.shape))
figure, axes = plt.subplots(2, 4, figsize=(10, 5))
for index, axis in enumerate(axes.flat):
    if index < len(batch_images):
        axis.imshow(batch_images[index].permute(1, 2, 0))  # Convert CHW to HWC for display.
        axis.set_title(class_names[int(batch_labels[index])])
    axis.axis("off")
plt.tight_layout()
plt.show()


### 4.7 Build a compact CNN baseline

Two Conv-ReLU-Pool blocks produce 32 feature maps of size 16 × 16 from a 64 × 64 input.
The classifier returns two raw logits. As in Session 12, CrossEntropyLoss receives logits
and integer targets; a separate softmax is unnecessary during training.


In [ ]:
from torch import nn

class CatDogCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),  # Read three RGB channels.
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),  # Reduce 64 × 64 to 32 × 32.
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),  # Reduce to 16 × 16.
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 16 * 16, 64),  # Match the flattened feature dimensions.
            nn.ReLU(),
            nn.Dropout(p=0.3),  # Randomly drop some activations during training.
            nn.Linear(64, 2),  # One output logit for each class.
        )

    def forward(self, images):
        return self.classifier(self.features(images))

model = CatDogCNN().to(device)
loss_function = nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(model.parameters(), lr=0.001)  # Set the optimisation learning rate.
print("Output shape:", tuple(model(batch_images.to(device)).shape))


### 4.8 Train and validate across epochs

The same function processes a complete loader. Only training records gradients and updates
weights. It averages loss and accuracy over samples, including a smaller final batch.
The checkpoint with the lowest validation loss is retained in memory.


In [ ]:
import copy

def run_epoch(loader, training):
    model.train(training)  # Enable dropout for training; disable it for evaluation.
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(training):  # Record gradients only during training.
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if training:
                optimiser.zero_grad()  # Clear gradients from the preceding batch.
            logits = model(images)
            loss = loss_function(logits, labels)
            if training:
                loss.backward()  # Compute parameter gradients.
                optimiser.step()  # Update parameters using those gradients.
            total_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total

history = {name: [] for name in ["train_loss", "val_loss", "train_accuracy", "val_accuracy"]}
best_val_loss = float("inf")
best_weights = None
for epoch in range(EPOCHS):
    train_loss, train_accuracy = run_epoch(loaders["train"], training=True)
    val_loss, val_accuracy = run_epoch(loaders["val"], training=False)
    for name, value in [("train_loss", train_loss), ("val_loss", val_loss),
                        ("train_accuracy", train_accuracy), ("val_accuracy", val_accuracy)]:
        history[name].append(value)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights = copy.deepcopy(model.state_dict())  # Preserve values before later updates.
        best_epoch = epoch + 1
    print(f"Epoch {epoch + 1}: train loss={train_loss:.4f}, val loss={val_loss:.4f}, "
          f"train accuracy={train_accuracy:.1%}, val accuracy={val_accuracy:.1%}")

model.load_state_dict(best_weights)  # Restore the checkpoint selected using validation data.
print("Selected epoch:", best_epoch)


### 4.9 Read the learning curves

Falling training loss with rising validation loss can indicate overfitting. If both losses
remain high, inspect labels, image preparation and model capacity. Three epochs may be too
few to draw a clear conclusion; use validation results to plan the next experiment.


In [ ]:
epoch_numbers = range(1, EPOCHS + 1)
figure, axes = plt.subplots(1, 2, figsize=(10, 3))
for name in ["train", "val"]:
    axes[0].plot(epoch_numbers, history[name + "_loss"], marker="o", label=name)
    axes[1].plot(epoch_numbers, history[name + "_accuracy"], marker="o", label=name)
for axis, title in zip(axes, ["Loss", "Accuracy"]):
    axis.set_title(title)
    axis.set_xlabel("Epoch")
    axis.legend()
axes[1].set_ylim(0, 1)
plt.tight_layout()
plt.show()


### 4.10 Evaluate the selected model on the test set

Compare test accuracy with the majority-class baseline. The confusion matrix uses true
classes as rows and predicted classes as columns. Per-class recall helps reveal whether one
class is missed more often. Do not use the test score to select another checkpoint.


In [ ]:
model.eval()  # Disable training-only behaviour.
test_targets, test_predictions = [], []
with torch.no_grad():  # Evaluation does not update weights.
    for images, labels in loaders["test"]:
        predictions = model(images.to(device)).argmax(dim=1).cpu()
        test_predictions.extend(predictions.tolist())
        test_targets.extend(labels.tolist())

confusion = np.zeros((2, 2), dtype=int)
for target, prediction in zip(test_targets, test_predictions):
    confusion[target, prediction] += 1
accuracy = np.trace(confusion) / confusion.sum()
baseline = max(test_targets.count(index) for index in range(2)) / len(test_targets)
print(f"Test accuracy: {accuracy:.1%}; majority-class baseline: {baseline:.1%}")
for index, name in enumerate(class_names):
    print(f"{name} recall: {confusion[index, index] / confusion[index].sum():.1%}")

figure, axis = plt.subplots(figsize=(4, 4))
axis.imshow(confusion, cmap="Blues")
axis.set(xticks=[0, 1], yticks=[0, 1], xticklabels=class_names, yticklabels=class_names,
         xlabel="Predicted class", ylabel="True class", title="Test confusion matrix")
for row in range(2):
    for column in range(2):
        axis.text(column, row, str(confusion[row, column]), ha="center", va="center")
plt.tight_layout()
plt.show()


### 4.11 Inspect predictions and mistakes

Show up to eight misclassified test images; if there are none, show the first eight test
examples. Softmax scores describe this model's relative scores for the two classes.
They are not a guarantee of calibrated confidence or correct handling of unfamiliar images.


In [ ]:
mistakes = [i for i, (actual, predicted) in enumerate(zip(test_targets, test_predictions))
            if actual != predicted]
examples = mistakes[:8] if mistakes else list(range(min(8, len(test_targets))))
figure, axes = plt.subplots(2, 4, figsize=(12, 6))
for axis in axes.flat:
    axis.axis("off")
with torch.no_grad():
    for axis, index in zip(axes.flat, examples):
        image, actual = image_datasets["test"][index]
        scores = model(image.unsqueeze(0).to(device)).softmax(dim=1)[0].cpu()
        predicted = int(scores.argmax())
        axis.imshow(image.permute(1, 2, 0))
        axis.set_title(f"True: {class_names[actual]} | Pred: {class_names[predicted]}\n"
                       f"Model score: {float(scores[predicted]):.2f}")
plt.tight_layout()
plt.show()


### 4.12 Practice and discussion

- Change one training setting, such as learning rate or epoch count. Compare validation curves
  on the same split before evaluating a final chosen model on the test set.
- Explain why the final Linear layer has two outputs and the first convolution has three input channels.
- Identify mistakes involving backgrounds, lighting, poses or image quality.
- Explain why a photo containing both a cat and a dog does not fit the one-label task well.

**Suggested answers:** output count matches the two classes; RGB provides three channels;
validation guides model selection while the test set estimates final generalisation.

**Optional extension:** revisit Session 12 transfer learning. Use the pretrained model's
documented resize and normalisation, replace its classification head and compare against this
baseline using the same split. Do not assume the current 64 × 64 pipeline is the correct
preprocessing for pretrained weights.

[Official tutorial: transfer learning for computer vision](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)

## 5. Object Detection: YOLO and DETR

### 5.1 From one label to object locations

Classification asks, “What class describes this image?” Detection asks, “Which objects are
present, and where are they?” A detector can return several bounding boxes, class labels and
scores for one image. Detection training requires location annotations as well as labels;
our class-folder dataset alone does not provide those boxes.

| Task | Example output for a photo with two animals |
|---|---|
| Classification | One whole-image label: dog |
| Object detection | A cat box and a dog box, each with a class and score |
| Image generation | A newly synthesised animal image |

### 5.2 YOLO

YOLO stands for “You Only Look Once”. The family predicts object locations and classes in a
single forward pass and is commonly used for real-time detection. Speed and accuracy depend
on the model variant, input size and hardware. A detection score is not a proof of correctness.

[Official documentation: Ultralytics YOLO](https://docs.ultralytics.com/)

### 5.3 DETR

DETR stands for “DEtection TRansformer”. The original architecture combines CNN image
features with a Transformer and learned object queries to predict a set of objects.
Training matches predicted objects to labelled boxes. Original DETR avoids the usual
hand-designed anchor and non-maximum-suppression pipeline.

[Original DETR paper](https://arxiv.org/abs/2005.12872)

**Discuss:** a single image shows three dogs and one cat. Which task supplies a location for
each animal? What extra labels would we need to train that model?

**Expected response:** object detection; a class and bounding box for each labelled animal.

## 6. Image Generation: GANs and Diffusion Models

### 6.1 Generative adversarial networks

A GAN trains two networks together. The generator maps a latent input to a synthetic image.
The discriminator learns to distinguish real training images from generated ones. Their
competing objectives encourage the generator to produce more realistic samples.

`Random latent input → generator → synthetic image → discriminator`

GAN training can be unstable, and a generator may produce a limited variety of outputs
(mode collapse). A basic GAN is not automatically controlled by a text prompt.

[Original GAN paper](https://arxiv.org/abs/1406.2661)

### 6.2 Diffusion models

During training, noise is added to images at different levels and a neural network learns
a denoising task. During sampling, a learned reverse process starts from noise and gradually
constructs an image. A text-conditioned model also uses a representation of a prompt to guide
generation. Not every diffusion model is text-conditioned.

`Noise → repeated learned denoising steps → generated image`

[Original DDPM paper](https://arxiv.org/abs/2006.11239)

### 6.3 Compare the ideas

| Method | Learning idea | Typical sampling pattern |
|---|---|---|
| GAN | Generator competes with a discriminator | A forward pass through the generator |
| Diffusion | Learn to reverse a noise process | A sequence of denoising steps |

Sampling speed and quality vary by implementation. Neither method guarantees factual images,
unbiased outputs or freedom from memorised training content. Review generated material before use.

**Check for understanding:**

1. Which network creates images in a GAN?
2. What does a diffusion model learn to reverse?
3. How does generation differ from assigning a cat/dog label?

**Expected responses:** the generator; a noise-adding process; generation synthesises a new
image, whereas classification predicts a label for an existing image.

**Exit task:** describe the complete image-classification workflow and give one example each
of classification, detection and generation.
